# SolarSDE 07a2 — Stanford SKIPP'D Pipeline (~6-8 hours)

**Run 07a1 FIRST.** This notebook needs the Golden artifacts from 07a1.

## What this notebook does

| Step | What | Time |
|------|------|------|
| Stanford SKIPP'D | Download HDF5 ~4.4 GB, train Stanford VAE 30 epochs, extract latents + CTI, train Stanford SDE + Score Decoder 30 epochs each, evaluate at horizons [1,5,10,20,30] min | ~6-8h |
| Zip outputs | Final cell zips PERSIST_DIR to /kaggle/working/ for download | <5 min |

## Kaggle workflow

1. Enable P100 GPU + Internet
2. **Attach 07a1's output** as a dataset input (Add Data → Your Datasets → select your 07a1 saved version)
3. Run all cells. The setup cell looks for Golden artifacts in standard locations.
4. Download `solarsde_outputs_combined.zip` from Output tab when done.


## 0. Setup

In [ ]:
# ==== Dependencies ====
import subprocess, sys
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
pip_install("pvlib", "properscoring", "pyarrow", "tqdm")

# ==== Environment detection ====
import os, time, json, shutil, gc, math
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")
print(f"Environment: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print(f"Drive mount issue: {e}")
    PERSIST_DIR = Path("/content/drive/MyDrive/solarsde_outputs")
    WORK_DIR = Path("/content/solarsde")
elif IN_KAGGLE:
    PERSIST_DIR = Path("/kaggle/working/solarsde_outputs")
    WORK_DIR = Path("/kaggle/working/solarsde")
else:
    PERSIST_DIR = Path.cwd() / "solarsde_outputs"
    WORK_DIR = Path.cwd() / "solarsde_work"

for d in [PERSIST_DIR, WORK_DIR,
          PERSIST_DIR / "checkpoints", PERSIST_DIR / "results",
          PERSIST_DIR / "latents",     PERSIST_DIR / "splits",
          PERSIST_DIR / "extended",    PERSIST_DIR / "figures"]:
    d.mkdir(parents=True, exist_ok=True)

DATA_DIR        = WORK_DIR / "data"
CHECKPOINT_DIR  = PERSIST_DIR / "checkpoints"
RESULTS_DIR     = PERSIST_DIR / "results"
LATENT_DIR      = PERSIST_DIR / "latents"
SPLITS_DIR      = PERSIST_DIR / "splits"
EXTENDED_DIR    = PERSIST_DIR / "extended"
FIGURES_DIR     = PERSIST_DIR / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Persistent storage: {PERSIST_DIR}")

# ==== Clean-start toggle (set True on FIRST Kaggle run or when switching to v2 arch) ====
# v1 checkpoints are incompatible with v2 architecture; clearing them forces retrain.
CLEAN_START_V2 = False    # flip to True ONCE if switching from v1 -> v2
if CLEAN_START_V2:
    print("CLEAN_START_V2=True: removing old v1 checkpoints + results ...")
    for f in ["checkpoints/sde_best.pt", "checkpoints/sde_final.pt",
              "checkpoints/score_best.pt", "checkpoints/score_final.pt",
              "checkpoints/sde_a2_best.pt", "checkpoints/sde_a5_best.pt",
              "checkpoints/linear_decoder_a4.pt",
              "results/solar_sde_main_results.csv",
              "results/main_results_combined.csv",
              "results/ablation_results.csv",
              "results/solar_sde_calibrated.csv"]:
        p = PERSIST_DIR / f
        if p.exists():
            p.unlink()
            print(f"  removed {p.name}")
    print("Clean slate ready — Stage 0 will retrain with v2.")

# ==== GPU setup ====
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader, TensorDataset
from tqdm.auto import tqdm

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device:  {torch.cuda.get_device_name(0)}")
    DEVICE = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
else:
    DEVICE = torch.device("cpu")
    print("WARNING: CPU only. This notebook needs a GPU; please enable one.")
print(f"Using device: {DEVICE}")


In [ ]:
# ==== Soft fast-start: pull cached artifacts from GitHub if available ====
# Never raises — if anything is missing, the from-scratch stages below will
# produce it. This block exists purely as a fast path for users who don't
# want to spend 6+ hours retraining the Golden VAE.

import requests
GITHUB_RAW = "https://raw.githubusercontent.com/keshavkrishnan08/SDE/main"

def gh_pull_soft(rel_path, dest):
    if dest.exists() and dest.stat().st_size > 100:
        return True
    try:
        r = requests.get(f"{GITHUB_RAW}/{rel_path}", timeout=180)
        if r.status_code == 200 and len(r.content) > 100:
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(r.content)
            return True
    except Exception:
        pass
    return False

print("Trying to pull cached upstream artifacts (best-effort, non-blocking) ...")
required = {
    CHECKPOINT_DIR / "vae_best.pt":   "colab_outputs/checkpoints/vae_best.pt",
    SPLITS_DIR    / "train.parquet":  "colab_outputs/splits/train.parquet",
    SPLITS_DIR    / "val.parquet":    "colab_outputs/splits/val.parquet",
    SPLITS_DIR    / "test.parquet":   "colab_outputs/splits/test.parquet",
    EXTENDED_DIR  / "train.parquet":  "colab_outputs/extended/train.parquet",
    EXTENDED_DIR  / "val.parquet":    "colab_outputs/extended/val.parquet",
    EXTENDED_DIR  / "test.parquet":   "colab_outputs/extended/test.parquet",
}
for split in ["train", "val", "test"]:
    for key in ["latents", "cti", "ghi", "covariates", "is_ramp", "kt", "ghi_clearsky",
                "physics_features"]:
        required[LATENT_DIR / f"{split}_{key}.npy"] = f"colab_outputs/latents/{split}_{key}.npy"
optional = {
    CHECKPOINT_DIR / "sde_best.pt":   "colab_outputs/checkpoints/sde_best.pt",
    CHECKPOINT_DIR / "score_best.pt": "colab_outputs/checkpoints/score_best.pt",
}
n_pulled, n_missing = 0, 0
for dest, rel in {**required, **optional}.items():
    if gh_pull_soft(rel, dest):
        if dest.exists():
            n_pulled += 1
    else:
        n_missing += 1

print(f"  pulled/already-present: {n_pulled}    missing: {n_missing}")
HAVE_VAE = (CHECKPOINT_DIR / "vae_best.pt").exists()
HAVE_SPLITS = (SPLITS_DIR / "train.parquet").exists()
HAVE_LATENTS = all((LATENT_DIR / f"{s}_latents.npy").exists() for s in ["train", "val", "test"])
HAVE_KT = all((LATENT_DIR / f"{s}_kt.npy").exists() for s in ["train", "val", "test"])
HAVE_PHYS = all((LATENT_DIR / f"{s}_physics_features.npy").exists() for s in ["train", "val", "test"])
HAVE_EXTENDED = (EXTENDED_DIR / "train.parquet").exists()

print(f"\nState of upstream artifacts:")
print(f"  VAE checkpoint:      {HAVE_VAE}")
print(f"  Splits parquets:     {HAVE_SPLITS}")
print(f"  Latents (z+cti):     {HAVE_LATENTS}")
print(f"  kt + ghi_clearsky:   {HAVE_KT}")
print(f"  Physics features:    {HAVE_PHYS}")
print(f"  Extended parquets:   {HAVE_EXTENDED}")

NEED_GOLDEN_RETRAIN = not (HAVE_VAE and HAVE_SPLITS and HAVE_LATENTS and HAVE_KT and HAVE_PHYS)
if NEED_GOLDEN_RETRAIN:
    print("\n[INFO] Some Golden artifacts missing — RETRAIN_GOLDEN stage will produce them.")
else:
    print("\n[INFO] All Golden artifacts present — RETRAIN_GOLDEN stage will skip.")


## 1. Shared model definitions

In [ ]:
# ==== Shared model definitions (matches Notebooks 1 + 2) ====

# --- CS-VAE (needed only for sanity; not retrained here) ---
class VAEEncoder(nn.Module):
    def __init__(self, latent_dim=64, channels=(32, 64, 128, 256)):
        super().__init__()
        layers, in_ch = [], 3
        for ch in channels:
            layers.extend([nn.Conv2d(in_ch, ch, 4, 2, 1),
                           nn.GroupNorm(min(32, ch), ch),
                           nn.SiLU(inplace=True)])
            in_ch = ch
        self.conv = nn.Sequential(*layers); self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc_mu = nn.Linear(channels[-1], latent_dim)
        self.fc_lv = nn.Linear(channels[-1], latent_dim)
    def forward(self, x):
        h = self.pool(self.conv(x)).flatten(1)
        return self.fc_mu(h), self.fc_lv(h)

# --- Neural SDE ---
class ResBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d), nn.SiLU(inplace=True), nn.Linear(d, d))
    def forward(self, x): return x + self.net(x)

class DriftNet(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, h=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim + 1 + c_dim, h), nn.SiLU(inplace=True),
            nn.Linear(h, h), nn.SiLU(inplace=True),
            ResBlock(h), ResBlock(h),
            nn.Linear(h, z_dim),
        )
    def forward(self, z, t, c): return self.net(torch.cat([z, t, c], dim=-1))

SIGMA_FLOOR_BASE = 0.01

class CTIDiffNet(nn.Module):
    """v2: diffusion floor + CTI scaling. sigma = floor(1+10*cti) + learned_softplus"""
    def __init__(self, z_dim=64, h=64, sigma_floor=SIGMA_FLOOR_BASE):
        super().__init__()
        self.sigma_floor = sigma_floor
        self.cti_gate = nn.Sequential(nn.Linear(1, h), nn.Softplus())
        self.state = nn.Sequential(nn.Linear(z_dim, h), nn.SiLU(inplace=True))
        self.out = nn.Sequential(nn.Linear(h, z_dim), nn.Softplus())
    def forward(self, z, cti):
        base_floor = self.sigma_floor * (1.0 + 10.0 * cti)
        learned = self.out(self.state(z) * self.cti_gate(cti))
        return base_floor + learned

class LatentNeuralSDE(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, drift_h=256, diff_h=64, lambda_sigma=1.0):
        super().__init__()
        self.z_dim = z_dim; self.lambda_sigma = lambda_sigma
        self.drift = DriftNet(z_dim, c_dim, drift_h)
        self.diffusion = CTIDiffNet(z_dim, diff_h)
    def forward(self, z, t, c, cti):
        return self.drift(z, t, c), self.diffusion(z, cti)
    def sde_matching_loss(self, z, zn, t, c, cti, dt=1.0):
        mu = self.drift(z, t, c); sigma = self.diffusion(z, cti)
        dz = (zn - z) / dt
        drift_l = F.mse_loss(mu, dz)
        # v2: log-space diffusion matching (well-conditioned, prevents sigma collapse)
        resid = (zn - z - mu * dt).pow(2) / dt + 1e-8
        log_diff_l = F.mse_loss(torch.log(sigma.pow(2) + 1e-8), torch.log(resid))
        return {"loss": drift_l + self.lambda_sigma * log_diff_l,
                "drift": drift_l, "diffusion": log_diff_l}

# --- Score Decoder v3 (RESIDUAL prediction: delta_kt = kt(t+h) - kt(t)) ---
#
# v2 predicted absolute kt(t+h). For stable conditions where kt(t+h) ≈ kt(t),
# the model had to learn a near-identity mapping — neural nets are bad at this.
#
# v3 predicts delta_kt = kt(t+h) - kt(t). Targets are concentrated near 0
# (most timesteps have small change). At sampling time, we add the sampled
# delta to the current kt to get the prediction:
#
#   kt(t+h)_predicted = kt(t)_observed + delta_kt_sampled
#   GHI(t+h) = kt(t+h)_predicted * ghi_clearsky(t+h)
#
# This is the persistence-anchored parameterization. Default behavior is
# "no change" (delta=0 = persistence). Model learns to deviate from
# persistence only when context says so.

GHI_SCALE = 1200.0
KT_SCALE = 1.5
DELTA_KT_SCALE = 1.0    # delta_kt typically in [-1.0, 1.0], rarely outside

class ScoreRes(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d), nn.SiLU(inplace=True), nn.Linear(d, d))
    def forward(self, x): return x + self.net(x)

class ScoreNet(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, h=256, blocks=2):
        super().__init__()
        # Inputs: (noisy_target, s, z, cti, c, kt_current)
        d_in = 1 + 1 + z_dim + 1 + c_dim + 1
        layers = [nn.Linear(d_in, h), nn.SiLU(inplace=True)]
        for _ in range(blocks): layers.append(ScoreRes(h))
        layers.append(nn.Linear(h, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, g, s, z, cti, c, kt_cur):
        return self.net(torch.cat([g, s, z, cti, c, kt_cur], dim=-1))

class CondScoreDecoder(nn.Module):
    """v3: predicts delta_kt with persistence anchoring.

    Default mode (predict_mode='delta'):
      target = kt(t+h) - kt(t)
      sample: kt(t+h) = kt(t) + delta_sampled
    Other modes (legacy):
      'kt'  : predicts kt(t+h) directly (v2)
      'ghi' : predicts GHI(t+h) directly (v1)
    """
    def __init__(self, z_dim=64, c_dim=5, h=256, blocks=2, steps=100, b0=1e-4, b1=0.02,
                 predict_mode='delta'):
        super().__init__()
        self.steps = steps
        self.predict_mode = predict_mode
        if predict_mode == 'delta':
            self.target_scale = DELTA_KT_SCALE
        elif predict_mode == 'kt':
            self.target_scale = KT_SCALE
        else:
            self.target_scale = GHI_SCALE
        self.score = ScoreNet(z_dim, c_dim, h, blocks)
        betas = torch.linspace(b0, b1, steps); alphas = 1 - betas
        ac = torch.cumprod(alphas, dim=0)
        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("alphas_cum", ac)
        self.register_buffer("sac", torch.sqrt(ac))
        self.register_buffer("s1mac", torch.sqrt(1 - ac))

    def _normalize(self, y):
        # For delta, scale [-DELTA_KT_SCALE, DELTA_KT_SCALE] -> [-1, 1]
        if self.predict_mode == 'delta':
            return y.clamp(-self.target_scale, self.target_scale) / self.target_scale
        else:
            return y / self.target_scale * 2.0 - 1.0
    def _denormalize(self, y):
        if self.predict_mode == 'delta':
            return y * self.target_scale
        else:
            return (y + 1.0) / 2.0 * self.target_scale

    def training_loss(self, kt_target, kt_current, z, cti, c):
        """Train on residual (or absolute, depending on mode)."""
        if self.predict_mode == 'delta':
            target_raw = kt_target - kt_current
        elif self.predict_mode == 'kt':
            target_raw = kt_target
        else:
            target_raw = kt_target  # caller passes ghi values in this mode
        t_norm = self._normalize(target_raw)
        B = t_norm.shape[0]
        si = torch.randint(0, self.steps, (B,), device=t_norm.device)
        sn = (si.float() / self.steps).unsqueeze(-1)
        eps = torch.randn_like(t_norm)
        ts = self.sac[si].unsqueeze(-1) * t_norm + self.s1mac[si].unsqueeze(-1) * eps
        kt_cur_in = kt_current.unsqueeze(-1) if kt_current.dim() == 1 else kt_current
        pred_noise = self.score(ts, sn, z, cti, c, kt_cur_in)
        return {"loss": F.mse_loss(pred_noise, eps)}

    @torch.no_grad()
    def sample(self, z, cti, c, kt_current, n=1):
        """Returns samples in kt-space.
           predict_mode='delta': returns kt(t+h) = kt_current + delta_sampled (clamped to [0, KT_SCALE])
           predict_mode='kt'   : returns kt(t+h) directly
           predict_mode='ghi'  : returns GHI(t+h) directly (caller doesn't multiply by gcs)
        """
        B = z.shape[0]
        z_e = z.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        cti_e = cti.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        c_e = c.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        kt_cur_in = kt_current.unsqueeze(-1) if kt_current.dim() == 1 else kt_current
        kt_cur_e = kt_cur_in.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        x = torch.randn(B * n, 1, device=z.device)
        for i in reversed(range(self.steps)):
            sn = torch.full((B * n, 1), i / self.steps, device=z.device)
            eps_pred = self.score(x, sn, z_e, cti_e, c_e, kt_cur_e)
            b, a, ac = self.betas[i], self.alphas[i], self.alphas_cum[i]
            mean = (1 / a.sqrt()) * (x - b / (1 - ac).sqrt() * eps_pred)
            if i > 0: x = mean + b.sqrt() * torch.randn_like(x)
            else:     x = mean
        y_unscaled = self._denormalize(x)   # in target space (delta_kt or kt or ghi)
        if self.predict_mode == 'delta':
            kt_out = (kt_cur_e + y_unscaled).clamp(0.0, KT_SCALE)
        elif self.predict_mode == 'kt':
            kt_out = y_unscaled.clamp(0.0, KT_SCALE)
        else:
            kt_out = y_unscaled.clamp(0.0, GHI_SCALE)
        return kt_out.view(B, n)

# --- Metrics ---
def crps_empirical(y_true, y_samples):
    """y_true: (N,), y_samples: (N, M). Returns per-point CRPS (N,)."""
    N, M = y_samples.shape
    t1 = np.mean(np.abs(y_samples - y_true[:, None]), axis=1)
    ys = np.sort(y_samples, axis=1)
    w = 2 * np.arange(1, M + 1) - M - 1
    t2 = np.sum(w[None, :] * ys, axis=1) / (M * M)
    return t1 - t2

def picp_metric(y_true, y_samples, alpha=0.9):
    lo = np.quantile(y_samples, (1 - alpha) / 2, axis=1)
    hi = np.quantile(y_samples, 1 - (1 - alpha) / 2, axis=1)
    return float(((y_true >= lo) & (y_true <= hi)).mean())

def pinaw_metric(y_samples, y_range, alpha=0.9):
    lo = np.quantile(y_samples, (1 - alpha) / 2, axis=1)
    hi = np.quantile(y_samples, 1 - (1 - alpha) / 2, axis=1)
    return float((hi - lo).mean() / max(y_range, 1e-9))

def all_metrics(y_true, y_samples, is_ramp=None, alpha=0.9):
    if len(y_true) == 0: return {"crps": 0, "picp": 0, "pinaw": 0, "rmse": 0, "mae": 0, "ramp_crps": 0}
    y_med = np.median(y_samples, axis=1)
    y_range = float(y_true.max() - y_true.min())
    crps = crps_empirical(y_true, y_samples)
    out = {
        "crps":  float(crps.mean()),
        "picp":  picp_metric(y_true, y_samples, alpha),
        "pinaw": pinaw_metric(y_samples, y_range, alpha),
        "rmse":  float(np.sqrt(np.mean((y_true - y_med) ** 2))),
        "mae":   float(np.mean(np.abs(y_true - y_med))),
    }
    if is_ramp is not None and is_ramp.sum() > 0:
        out["ramp_crps"] = float(crps[is_ramp].mean())
    else:
        out["ramp_crps"] = 0.0
    return out

# --- SDE solver (with stability clamping) ---
_train_Z_np = np.load(LATENT_DIR / "train_latents.npy")
Z_MEAN = torch.from_numpy(_train_Z_np.mean(0)).float().to(DEVICE)
Z_STD_RAW = torch.from_numpy(_train_Z_np.std(0)).float().to(DEVICE) + 1e-6
Z_STD = torch.maximum(Z_STD_RAW, torch.full_like(Z_STD_RAW, 0.05))
Z_CLAMP_STDS = 8.0
MU_CAP = 10.0
SIGMA_CAP = 5.0
del _train_Z_np

def em_step(drift_fn, diff_fn, z, t, c, cti, dt):
    mu = drift_fn(z, t, c).clamp(-MU_CAP, MU_CAP)
    sigma = diff_fn(z, cti).clamp(0.0, SIGMA_CAP)
    z_new = z + mu * dt + sigma * (dt ** 0.5) * torch.randn_like(z)
    return torch.clamp(z_new, Z_MEAN - Z_CLAMP_STDS * Z_STD, Z_MEAN + Z_CLAMP_STDS * Z_STD)

def solve_sde_horizons(sde, z0, horizons, c, cti, N=50, dt=1.0):
    """v4: with mixed-horizon training, the drift takes (z, normalized_horizon, c).
    At inference, we pass normalized_horizon = current_step / MAX_HORIZON as time input.
    This matches how the SDE was trained (drift(z, k/180, c) -> dz/k).
    The EM step uses physical dt=1.0; drift output is already in per-step units.
    """
    B, d = z0.shape
    mx = max(horizons); hset = set(horizons)
    MAX_HORIZON = 180.0
    z = z0.unsqueeze(1).expand(B, N, d).reshape(B * N, d)
    c_e = c.unsqueeze(1).expand(B, N, -1).reshape(B * N, -1)
    cti_e = cti.unsqueeze(1).expand(B, N, -1).reshape(B * N, -1)
    out = {}
    for step in range(mx):
        t_norm = torch.full((B * N, 1), (step + 1) / MAX_HORIZON, device=z0.device)
        z = em_step(sde.drift, sde.diffusion, z, t_norm, c_e, cti_e, dt)
        if (step + 1) in hset: out[step + 1] = z.view(B, N, d).clone()
    return out

print("Shared code loaded.")


## Prerequisite check

In [ ]:
# ==== Check that Part 1 (Foundations) artifacts are available ====
_missing = []
for p in [CHECKPOINT_DIR / "vae_best.pt",
          LATENT_DIR / "test_latents.npy",
          LATENT_DIR / "test_kt.npy",
          LATENT_DIR / "test_physics_features.npy",
          SPLITS_DIR / "test.parquet"]:
    if not p.exists():
        _missing.append(str(p))
if _missing:
    msg = ("This notebook expects Part 1 (07a_foundations) to have been run first.\n"
           "Missing artifacts:\n  " + "\n  ".join(_missing) +
           "\n\nOn Kaggle: go to Add Data, attach the output dataset from your 07a run.\n"
           "Locally: re-run 07a_foundations.ipynb first.")
    raise RuntimeError(msg)
print("Part 1 artifacts found — ready to proceed.")


## 2. Load data tensors

In [ ]:
# ==== Load all data tensors (tolerant: degrades gracefully if extended missing) ====
def load_split(s):
    orig_cov = np.load(LATENT_DIR / f"{s}_covariates.npy")
    phys = np.load(LATENT_DIR / f"{s}_physics_features.npy")
    img_feat_path = LATENT_DIR / f"{s}_image_features.npy"
    if img_feat_path.exists():
        img_feats = np.load(img_feat_path)
        cov = np.concatenate([orig_cov, phys, img_feats], axis=1).astype(np.float32)
    else:
        cov = np.concatenate([orig_cov, phys], axis=1).astype(np.float32)
    return {
        "Z":    np.load(LATENT_DIR / f"{s}_latents.npy"),
        "cti":  np.load(LATENT_DIR / f"{s}_cti.npy"),
        "ghi":  np.load(LATENT_DIR / f"{s}_ghi.npy"),
        "cov":  cov,
        "ramp": np.load(LATENT_DIR / f"{s}_is_ramp.npy"),
        "kt":   np.load(LATENT_DIR / f"{s}_kt.npy"),
        "gcs":  np.load(LATENT_DIR / f"{s}_ghi_clearsky.npy"),
    }
data = {s: load_split(s) for s in ["train", "val", "test"]}
print(f"\n  Covariate dim: {data['train']['cov'].shape[1]}  "
      f"(5 original + 15 physics + "
      f"{data['train']['cov'].shape[1] - 20} image features)")
for s, d in data.items():
    print(f"  {s}: Z={d['Z'].shape}, GHI=[{d['ghi'].min():.0f},{d['ghi'].max():.0f}], ramps={int(d['ramp'].sum())}")

train_df = pd.read_parquet(SPLITS_DIR / "train.parquet")
val_df   = pd.read_parquet(SPLITS_DIR / "val.parquet")
test_df  = pd.read_parquet(SPLITS_DIR / "test.parquet")
print(f"\n8-day image splits: train={len(train_df):,} val={len(val_df):,} test={len(test_df):,}")

# Extended (90-day BMS) splits — used by LSTM/MC-Dropout/TimeGrad/Deep-Ensemble baselines.
# If missing, we fall back to using the regular train_df/val_df for those baselines.
HAVE_EXT = (EXTENDED_DIR / "train.parquet").exists() and (EXTENDED_DIR / "val.parquet").exists()
if HAVE_EXT:
    ext_train = pd.read_parquet(EXTENDED_DIR / "train.parquet")
    ext_val   = pd.read_parquet(EXTENDED_DIR / "val.parquet")
    print(f"90-day extended:    train={len(ext_train):,} val={len(ext_val):,}")
else:
    print("[WARN] Extended (90-day BMS) parquets missing — LSTM baselines will train on the")
    print("       8-day image splits instead, with reduced sample count.")
    # Fallback: replicate the structure expected by BASELINES_CODE
    ext_train = train_df.copy()
    ext_val   = val_df.copy()

Z_DIM = data["train"]["Z"].shape[1]
C_DIM = max(1, data["train"]["cov"].shape[1])
print(f"\nZ_DIM={Z_DIM}, C_DIM={C_DIM}")

HORIZONS = [6, 30, 60, 120, 180]
HORIZON_MIN = {6: 1, 30: 5, 60: 10, 120: 20, 180: 30}
N_SAMPLES = 50
# Larger N_EVAL gives tighter bootstrap CIs. 2000 is ~12% of typical test set,
# enough for ramp events to be represented at expected ~5-10% rate.
N_EVAL = min(2000, len(data["test"]["Z"]) - max(HORIZONS) - 1)
SEQ_LEN = 30
print(f"Horizons: {list(HORIZON_MIN.values())} min, MC samples: {N_SAMPLES}, N_EVAL: {N_EVAL}")


## STAGE A — Stanford SKIPP'D as full second site

In [ ]:
# ==== STAGE A: Stanford SKIPP'D as full second site ====
# Downloads the SKIPP'D HDF5, trains a separate VAE on its 64x64 images
# (upsampled to 128x128), trains a separate SDE+Score Decoder on Stanford's
# PV power as the forecast target, and runs forecast evaluation.
#
# Stanford SKIPP'D: 497 trainval days + ~100 test days at 1-min resolution.
# Target = PV power (kW from one panel). NOT GHI.
# This gives us a SECOND independent experimental result for the paper.

ENABLE_STANFORD = True   # set False to skip Stanford pipeline entirely

SF_DIR = WORK_DIR / "stanford_skippd"
SF_DIR.mkdir(parents=True, exist_ok=True)
SF_HDF5 = SF_DIR / "2017_2019_images_pv_processed.hdf5"
SF_TIMES_TV = SF_DIR / "times_trainval.npy"
SF_TIMES_TE = SF_DIR / "times_test.npy"

SF_VAE_CKPT = CHECKPOINT_DIR / "stanford_vae_best.pt"
SF_SDE_CKPT = CHECKPOINT_DIR / "stanford_sde_best.pt"
SF_SCORE_CKPT = CHECKPOINT_DIR / "stanford_score_best.pt"
SF_LATENTS_DIR = LATENT_DIR / "stanford"
SF_LATENTS_DIR.mkdir(parents=True, exist_ok=True)
SF_RESULTS_DIR = RESULTS_DIR / "stanford"
SF_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

STAGE_A_DONE = (SF_RESULTS_DIR / "solarsde_results.csv").exists()

if not ENABLE_STANFORD:
    print("[SKIP] Stage A disabled (ENABLE_STANFORD=False).")
elif STAGE_A_DONE:
    print("[SKIP] Stage A: Stanford pipeline already complete (results CSV exists).")
else:
    print("=" * 70)
    print("STAGE A: Stanford SKIPP'D full pipeline")
    print("=" * 70)
    pip_install("h5py")
    import h5py

    # ---- A.1 Download SKIPP'D HDF5 (~3-4 GB) with retry + size validation ----
    SF_HDF5_MIN_SIZE = 3_000_000_000   # actual file ~4.4 GB; reject anything <3 GB
    import requests, time as _time

    def _sf_download(url, dest, min_size, max_retries=4, chunk=1024 * 1024):
        if dest.exists() and dest.stat().st_size >= min_size:
            print(f"  Already have: {dest.name} ({dest.stat().st_size / 1e9:.2f} GB)")
            return True
        if dest.exists() and dest.stat().st_size < min_size:
            print(f"  Partial/corrupt {dest.name} ({dest.stat().st_size / 1e6:.2f} MB) — removing.")
            dest.unlink()
        last_err = None
        for attempt in range(1, max_retries + 1):
            try:
                print(f"  [attempt {attempt}/{max_retries}] {dest.name}")
                with requests.get(url, stream=True, timeout=3600) as r:
                    r.raise_for_status()
                    total = int(r.headers.get("content-length", 0))
                    with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=dest.name) as pb:
                        for c in r.iter_content(chunk_size=chunk):
                            if c:
                                f.write(c); pb.update(len(c))
                if dest.stat().st_size < min_size:
                    raise RuntimeError(f"got {dest.stat().st_size / 1e6:.2f} MB < min {min_size / 1e6:.0f} MB")
                return True
            except Exception as e:
                last_err = e
                print(f"    FAILED: {e}")
                if dest.exists(): dest.unlink()
                if attempt < max_retries:
                    _time.sleep(5 * attempt)
        raise RuntimeError(f"Stanford download {dest.name} failed after {max_retries} attempts: {last_err}")

    print("[A.1] Downloading SKIPP'D HDF5 (~4.4 GB) ...")
    _sf_download("https://stacks.stanford.edu/file/dj417rh1007/2017_2019_images_pv_processed.hdf5",
                 SF_HDF5, SF_HDF5_MIN_SIZE)
    # Times metadata (small files, simpler retry)
    for nm, min_b in [("times_trainval.npy", 1_000_000), ("times_test.npy", 100_000)]:
        dst = SF_DIR / nm
        if dst.exists() and dst.stat().st_size >= min_b:
            continue
        _sf_download(f"https://stacks.stanford.edu/file/dj417rh1007/{nm}", dst, min_b, max_retries=3)

    # ---- A.2 Load splits + persist as npy ----
    print("[A.2] Loading SKIPP'D HDF5 ...")
    with h5py.File(SF_HDF5, "r") as f:
        for split_key, prefix in [("trainval", "stanford_train"), ("test", "stanford_test")]:
            if split_key not in f:
                continue
            grp = f[split_key]
            img_key = "images_log" if "images_log" in grp else list(grp.keys())[0]
            pv_key = "pv_log" if "pv_log" in grp else next(k for k in grp.keys() if "pv" in k.lower())
            np.save(SF_DIR / f"{prefix}_images.npy", grp[img_key][:])
            np.save(SF_DIR / f"{prefix}_pv.npy", grp[pv_key][:])
    print("  saved per-split npy files")

    # Build train/val/test splits with timestamps
    sf_train_imgs = np.load(SF_DIR / "stanford_train_images.npy")
    sf_train_pv = np.load(SF_DIR / "stanford_train_pv.npy")
    sf_train_times = np.load(SF_DIR / "times_trainval.npy", allow_pickle=True)
    sf_test_imgs = np.load(SF_DIR / "stanford_test_images.npy")
    sf_test_pv = np.load(SF_DIR / "stanford_test_pv.npy")
    sf_test_times = np.load(SF_DIR / "times_test.npy", allow_pickle=True)

    # 80/20 split of trainval into train/val (chronological by date)
    import pandas as pd
    ts_tv = pd.to_datetime(sf_train_times)
    days_tv = sorted(set(ts_tv.normalize()))
    n_train_days = int(len(days_tv) * 0.8)
    train_day_set = set(days_tv[:n_train_days])
    train_mask = np.array([t.normalize() in train_day_set for t in ts_tv])
    val_mask = ~train_mask
    print(f"  Stanford: train={train_mask.sum()} val={val_mask.sum()} test={len(sf_test_pv)}")

    SF_PV_SCALE = float(np.percentile(sf_train_pv, 99))   # use 99th percentile for normalization
    print(f"  PV scale (99th pct): {SF_PV_SCALE:.2f} kW")
    np.save(SF_LATENTS_DIR / "pv_scale.npy", np.array([SF_PV_SCALE]))

    # ---- VAE architecture (inline, matches Notebook 1's STAGE_MINUS2 VAE) ----
    class _SfEnc(nn.Module):
        def __init__(self, latent=64, ch=(32, 64, 128, 256)):
            super().__init__(); L, ic = [], 3
            for c in ch:
                L += [nn.Conv2d(ic, c, 4, 2, 1), nn.GroupNorm(min(32, c), c), nn.SiLU(inplace=True)]
                ic = c
            self.conv = nn.Sequential(*L); self.pool = nn.AdaptiveAvgPool2d(1)
            self.fc_mu = nn.Linear(ch[-1], latent); self.fc_lv = nn.Linear(ch[-1], latent)
        def forward(self, x):
            h = self.pool(self.conv(x)).flatten(1); return self.fc_mu(h), self.fc_lv(h)
    class _SfDec(nn.Module):
        def __init__(self, latent=64, ch=(256, 128, 64, 32)):
            super().__init__(); self.init_ch = ch[0]
            self.fc = nn.Linear(latent, ch[0] * 8 * 8); L = []
            for i in range(len(ch) - 1):
                L += [nn.ConvTranspose2d(ch[i], ch[i+1], 4, 2, 1),
                      nn.GroupNorm(min(32, ch[i+1]), ch[i+1]), nn.SiLU(inplace=True)]
            L += [nn.ConvTranspose2d(ch[-1], 3, 4, 2, 1), nn.Sigmoid()]
            self.deconv = nn.Sequential(*L)
        def forward(self, z): return self.deconv(self.fc(z).view(-1, self.init_ch, 8, 8))
    class _SfVAE(nn.Module):
        def __init__(self, latent=64, beta=0.1):
            super().__init__(); self.beta = beta
            self.encoder = _SfEnc(latent); self.decoder = _SfDec(latent)
        def forward(self, x):
            mu, lv = self.encoder(x); z = mu + torch.exp(0.5 * lv) * torch.randn_like(mu)
            return self.decoder(z), mu, lv
        def loss(self, x, rec, mu, lv):
            r = F.mse_loss(rec, x); k = -0.5 * torch.mean(1 + lv - mu.pow(2) - lv.exp())
            return r + self.beta * k

    # ---- A.3 Train Stanford VAE (independent) ----
    if not SF_VAE_CKPT.exists():
        print("[A.3] Training Stanford VAE (30 epochs, 128x128 upsampled) ...")
        class SfVAEDS(Dataset):
            def __init__(self, imgs, target=128):
                self.imgs = imgs; self.target = target
            def __len__(self): return len(self.imgs)
            def __getitem__(self, i):
                img = torch.from_numpy(self.imgs[i]).float() / 255.0
                if img.dim() == 3: img = img.permute(2, 0, 1)
                img = img.unsqueeze(0)
                img = F.interpolate(img, size=self.target, mode="bilinear", align_corners=False)
                return img.squeeze(0)
        ds = SfVAEDS(sf_train_imgs[train_mask])
        dl = DataLoader(ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
        torch.manual_seed(42)
        vae = _SfVAE(latent=64, beta=0.1).to(DEVICE)
        opt = torch.optim.Adam(vae.parameters(), lr=1e-4)
        best = float("inf")
        for ep in range(1, 31):
            vae.train(); tl = 0; n = 0
            for img in dl:
                img = img.to(DEVICE, non_blocking=True)
                recon, mu, lv = vae(img)
                loss = vae.loss(img, recon, mu, lv)
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0); opt.step()
                tl += loss.item(); n += 1
            tl /= n
            print(f"  ep {ep}/30: loss={tl:.4f}")
            if tl < best:
                best = tl; torch.save(vae.state_dict(), SF_VAE_CKPT)
        del vae, ds, dl; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    else:
        print("[A.3] Stanford VAE checkpoint exists, skipping.")

    # ---- A.4 Encode Stanford latents + CTI per split ----
    if not (SF_LATENTS_DIR / "test_latents.npy").exists():
        print("[A.4] Encoding Stanford latents + CTI ...")
        vae = _SfVAE(latent=64).to(DEVICE)
        vae.load_state_dict(torch.load(SF_VAE_CKPT, map_location=DEVICE, weights_only=False))
        vae.eval()

        @torch.no_grad()
        def encode_imgs(imgs, batch=128):
            out = []
            for i in tqdm(range(0, len(imgs), batch), desc="enc"):
                b = imgs[i:i+batch]
                x = torch.from_numpy(b).float() / 255.0
                if x.dim() == 4: x = x.permute(0, 3, 1, 2)
                x = F.interpolate(x, size=128, mode="bilinear", align_corners=False).to(DEVICE)
                mu, _ = vae.encoder(x)
                out.append(mu.cpu().numpy())
            return np.concatenate(out, axis=0)

        def cti_window(z, w=10):
            n = len(z); cti = np.zeros(n, dtype=np.float32)
            for i in range(w, n):
                v = np.diff(z[i-w:i+1], axis=0)
                cti[i] = np.linalg.norm(np.var(v, axis=0))
            return cti

        for sp_name, imgs, mask, pv, times in [
            ("train", sf_train_imgs[train_mask], None, sf_train_pv[train_mask], ts_tv[train_mask]),
            ("val",   sf_train_imgs[val_mask],   None, sf_train_pv[val_mask],   ts_tv[val_mask]),
            ("test",  sf_test_imgs,              None, sf_test_pv,               pd.to_datetime(sf_test_times)),
        ]:
            print(f"  encoding {sp_name} ({len(imgs)} samples)")
            z = encode_imgs(imgs)
            cti = cti_window(z, w=10)
            np.save(SF_LATENTS_DIR / f"{sp_name}_latents.npy", z)
            np.save(SF_LATENTS_DIR / f"{sp_name}_cti.npy", cti)
            np.save(SF_LATENTS_DIR / f"{sp_name}_pv.npy", pv.astype(np.float32))
            # Stanford has no GHI, so use PV/PV_scale as the equivalent of "kt"
            kt_proxy = (pv / SF_PV_SCALE).astype(np.float32)
            np.save(SF_LATENTS_DIR / f"{sp_name}_kt.npy", kt_proxy)
            # Minimal covariates: hour_sin/cos, doy_sin/cos, kt itself (autoregressive)
            ts = times
            hf = (ts.hour + ts.minute / 60.0).values
            doy = ts.dayofyear.values
            cov = np.stack([
                np.sin(2*np.pi*hf/24).astype(np.float32),
                np.cos(2*np.pi*hf/24).astype(np.float32),
                np.sin(2*np.pi*doy/365.25).astype(np.float32),
                np.cos(2*np.pi*doy/365.25).astype(np.float32),
                kt_proxy,
            ], axis=1)
            np.save(SF_LATENTS_DIR / f"{sp_name}_covariates.npy", cov)
        del vae; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    # ---- A.5 Train Stanford SDE + Score Decoder ----
    if not SF_SCORE_CKPT.exists():
        print("[A.5] Training Stanford SDE + Score Decoder ...")
        sf_z_tr = np.load(SF_LATENTS_DIR / "train_latents.npy")
        sf_cti_tr = np.load(SF_LATENTS_DIR / "train_cti.npy")
        sf_kt_tr = np.load(SF_LATENTS_DIR / "train_kt.npy")
        sf_cov_tr = np.load(SF_LATENTS_DIR / "train_covariates.npy")
        sf_z_val = np.load(SF_LATENTS_DIR / "val_latents.npy")
        sf_cti_val = np.load(SF_LATENTS_DIR / "val_cti.npy")
        sf_kt_val = np.load(SF_LATENTS_DIR / "val_kt.npy")
        sf_cov_val = np.load(SF_LATENTS_DIR / "val_covariates.npy")

        # SDE — same architecture, dt=60s for Stanford (1-min sampling)
        sf_sde = LatentNeuralSDE(z_dim=64, c_dim=sf_cov_tr.shape[1]).to(DEVICE)
        opt_sde = torch.optim.Adam(sf_sde.parameters(), lr=5e-4)

        # Mixed-horizon dataset
        class SfMHDS(Dataset):
            def __init__(self, z, cti, c, hs=(1, 5, 10, 15, 30), seed=42):
                self.z = z; self.cti = cti; self.c = c; self.hs = hs
                self.rng = np.random.RandomState(seed)
                self.maxh = max(hs)
                self.idx = np.arange(len(z) - self.maxh)
            def __len__(self): return len(self.idx)
            def __getitem__(self, i):
                ii = self.idx[i]; k = int(self.rng.choice(self.hs))
                return {
                    "z_t": torch.from_numpy(self.z[ii]),
                    "z_next": torch.from_numpy(self.z[ii + k]),
                    "k": torch.tensor(k, dtype=torch.float32),
                    "cti_t": torch.tensor(self.cti[ii], dtype=torch.float32),
                    "c_t": torch.from_numpy(self.c[ii]),
                }
        ds = SfMHDS(sf_z_tr, sf_cti_tr, sf_cov_tr)
        dl = DataLoader(ds, batch_size=512, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

        print(f"  SDE: training on {len(ds)} mixed-horizon transitions")
        for ep in range(1, 31):
            sf_sde.train(); tl_d = tl_s = 0; n = 0
            for b in dl:
                z = b["z_t"].to(DEVICE); zn = b["z_next"].to(DEVICE)
                k = b["k"].float().unsqueeze(-1).to(DEVICE)
                t = (k / 30.0)
                cti = b["cti_t"].unsqueeze(-1).to(DEVICE)
                c = b["c_t"].to(DEVICE)
                mu = sf_sde.drift(z, t, c)
                sigma = sf_sde.diffusion(z, cti)
                dz = (zn - z) / k
                drift_loss = F.mse_loss(mu, dz)
                resid = zn - z - mu * k
                target_var = (resid ** 2) / k.clamp(min=1.0)
                sigma_sq = sigma.pow(2).clamp(min=1e-6)
                diff_loss = F.mse_loss(torch.log(sigma_sq + 1e-8), torch.log(target_var + 1e-8))
                loss = drift_loss + 0.5 * diff_loss
                opt_sde.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(sf_sde.parameters(), 1.0); opt_sde.step()
                tl_d += drift_loss.item(); tl_s += diff_loss.item(); n += 1
            print(f"  SDE ep {ep}/30: drift={tl_d/n:.4f} diff={tl_s/n:.4f}")
        torch.save(sf_sde.state_dict(), SF_SDE_CKPT)

        # Score Decoder for Stanford (predict delta-kt over kt-proxy = PV/PV_scale)
        sf_score = CondScoreDecoder(z_dim=64, c_dim=sf_cov_tr.shape[1], predict_mode='delta').to(DEVICE)
        opt_score = torch.optim.Adam(sf_score.parameters(), lr=1e-4)

        class SfScoreDS(Dataset):
            def __init__(self, z, cti, c, kt, hs=(1, 5, 10, 15, 30), seed=42):
                self.z = z; self.cti = cti; self.c = c; self.kt = kt; self.hs = hs
                self.rng = np.random.RandomState(seed)
                self.maxh = max(hs)
            def __len__(self): return len(self.z) - self.maxh
            def __getitem__(self, i):
                k = int(self.rng.choice(self.hs))
                return {
                    "kt_target": torch.tensor(self.kt[i + k], dtype=torch.float32),
                    "kt_current": torch.tensor(self.kt[i], dtype=torch.float32),
                    "z_t": torch.from_numpy(self.z[i]),
                    "cti_t": torch.tensor(self.cti[i], dtype=torch.float32),
                    "c_t": torch.from_numpy(self.c[i]),
                }
        score_ds = SfScoreDS(sf_z_tr, sf_cti_tr, sf_cov_tr, sf_kt_tr)
        score_dl = DataLoader(score_ds, batch_size=512, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

        for ep in range(1, 31):
            sf_score.train(); tl = 0; n = 0
            for b in score_dl:
                kt_t = b["kt_target"].unsqueeze(-1).to(DEVICE)
                kt_c = b["kt_current"].unsqueeze(-1).to(DEVICE)
                z = b["z_t"].to(DEVICE); cti = b["cti_t"].unsqueeze(-1).to(DEVICE)
                c = b["c_t"].to(DEVICE)
                loss = sf_score.training_loss(kt_t, kt_c, z, cti, c)
                opt_score.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(sf_score.parameters(), 1.0); opt_score.step()
                tl += loss.item(); n += 1
            print(f"  Score ep {ep}/30: loss={tl/n:.4f}")
        torch.save(sf_score.state_dict(), SF_SCORE_CKPT)
        del sf_sde, sf_score; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    else:
        print("[A.5] Stanford SDE + Score checkpoints exist, skipping.")

    # ---- A.6 Evaluate Stanford SolarSDE on test set ----
    print("[A.6] Evaluating Stanford SolarSDE on test set ...")
    sf_z_te = np.load(SF_LATENTS_DIR / "test_latents.npy")
    sf_cti_te = np.load(SF_LATENTS_DIR / "test_cti.npy")
    sf_pv_te = np.load(SF_LATENTS_DIR / "test_pv.npy")
    sf_kt_te = np.load(SF_LATENTS_DIR / "test_kt.npy")
    sf_cov_te = np.load(SF_LATENTS_DIR / "test_covariates.npy")

    sf_sde = LatentNeuralSDE(z_dim=64, c_dim=sf_cov_te.shape[1]).to(DEVICE)
    sf_sde.load_state_dict(torch.load(SF_SDE_CKPT, map_location=DEVICE, weights_only=False))
    sf_sde.eval()
    sf_score = CondScoreDecoder(z_dim=64, c_dim=sf_cov_te.shape[1], predict_mode='delta').to(DEVICE)
    sf_score.load_state_dict(torch.load(SF_SCORE_CKPT, map_location=DEVICE, weights_only=False))
    sf_score.eval()

    HORIZONS_SF = [1, 5, 10, 20, 30]   # minutes — harmonized with Golden HORIZON_MIN values
    N_SAMPLES = 50
    rows = []
    with torch.no_grad():
        for h in HORIZONS_SF:
            preds_all = []
            truths = []
            for i in tqdm(range(0, len(sf_z_te) - h, 4), desc=f"h={h}"):
                z0 = torch.from_numpy(sf_z_te[i]).unsqueeze(0).repeat(N_SAMPLES, 1).to(DEVICE)
                cti0 = torch.tensor(sf_cti_te[i]).unsqueeze(0).unsqueeze(-1).repeat(N_SAMPLES, 1).to(DEVICE)
                c0 = torch.from_numpy(sf_cov_te[i]).unsqueeze(0).repeat(N_SAMPLES, 1).to(DEVICE)
                kt0 = torch.tensor(sf_kt_te[i]).unsqueeze(0).unsqueeze(-1).repeat(N_SAMPLES, 1).to(DEVICE)
                # Roll latent forward h steps via Euler-Maruyama
                z = z0
                for s in range(h):
                    t_norm = torch.full((N_SAMPLES, 1), float(s) / 30.0, device=DEVICE)
                    mu = sf_sde.drift(z, t_norm, c0)
                    sigma = sf_sde.diffusion(z, cti0)
                    z = z + mu * 1.0 + sigma * torch.randn_like(z)
                # Decode kt at horizon
                kt_pred = sf_score.sample(z, cti0, c0, kt0, n=1).squeeze(-1).cpu().numpy()
                pv_pred = kt_pred * SF_PV_SCALE
                preds_all.append(pv_pred)
                truths.append(sf_pv_te[i + h])
            preds = np.array(preds_all)         # (N_obs, N_samples)
            tru = np.array(truths)              # (N_obs,)
            crps = crps_empirical(tru, preds).mean()
            rmse = np.sqrt(((preds.mean(1) - tru) ** 2).mean())
            picp = ((np.percentile(preds, 5, axis=1) <= tru) &
                    (tru <= np.percentile(preds, 95, axis=1))).mean()
            pinaw = (np.percentile(preds, 95, axis=1) - np.percentile(preds, 5, axis=1)).mean() / max(tru.max() - tru.min(), 1.0)
            print(f"  h={h}: CRPS={crps:.3f}  RMSE={rmse:.3f}  PICP={picp:.3f}  PINAW={pinaw:.3f}")
            rows.append({"horizon_min": h, "crps": crps, "rmse": rmse, "picp": picp, "pinaw": pinaw})
            np.save(SF_RESULTS_DIR / f"solarsde_preds_h{h}.npy", preds)
            np.save(SF_RESULTS_DIR / f"truths_h{h}.npy", tru)
    pd.DataFrame(rows).to_csv(SF_RESULTS_DIR / "solarsde_results.csv", index=False)
    print(f"\nStanford SolarSDE results -> {SF_RESULTS_DIR / 'solarsde_results.csv'}")
    del sf_sde, sf_score; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


## Final: zip outputs for Kaggle download

In [ ]:
# ==== Zip and download all outputs ====
import shutil
zip_path = WORK_DIR / "solarsde_outputs_combined.zip"
if zip_path.exists(): zip_path.unlink()
print(f"Zipping {PERSIST_DIR} ...")
shutil.make_archive(str(zip_path).replace(".zip", ""), "zip", root_dir=PERSIST_DIR)
size_mb = zip_path.stat().st_size / 1e6
print(f"Archive: {zip_path}  ({size_mb:.1f} MB)")

if IN_COLAB:
    from google.colab import files
    try: files.download(str(zip_path))
    except Exception as e: print(f"Auto-download failed: {e}. File at {zip_path}")
elif IN_KAGGLE:
    # Copy the zip + the two headline CSVs to /kaggle/working/ top-level
    # so they show up prominently in the Output tab of the notebook.
    top = Path("/kaggle/working")
    shutil.copy(zip_path, top / zip_path.name)
    for key_file in ["results/main_results_combined.csv",
                     "results/ablation_results.csv",
                     "results/solar_sde_calibrated.csv"]:
        src = PERSIST_DIR / key_file
        if src.exists():
            shutil.copy(src, top / src.name)
    print(f"Kaggle: zip + key CSVs copied to /kaggle/working/. Use Output tab to download,")
    print(f"or 'Save Version' to commit them permanently as notebook outputs.")
else:
    print(f"Local: file at {zip_path}")

print("\n" + "=" * 70)
print("ALL STAGES COMPLETE")
print("=" * 70)
for sub in ["splits", "extended", "checkpoints", "latents", "results", "figures"]:
    p = PERSIST_DIR / sub
    if p.exists():
        n = sum(1 for _ in p.rglob("*") if _.is_file())
        total = sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
        print(f"  {sub}/: {n} files, {total/1e6:.1f} MB")
